<a href="https://colab.research.google.com/github/mhowlin-web/TP_RAG_ARCA/blob/main/01_descarga_corpus_arca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP RAG y Agentes

## Asistente para consultas sobre trámites de Monotributo en ARCA

En este proyecto voy a construir un sistema RAG para responder preguntas sobre
trámites relacionados con el Monotributo utilizando información proveniente de
fuentes oficiales de ARCA.

El sistema buscará información relevante dentro de un conjunto de documentos
oficiales y utilizará un modelo de lenguaje para generar respuestas basadas
únicamente en los documentos recuperados.

En este primer notebook construyo el corpus documental que utilizaré en las
siguientes etapas del proyecto.

El proceso que realizo en este notebook es:

1. Defino las fuentes oficiales.
2. Descargo automáticamente las páginas.
3. Extraigo el contenido textual.
4. Realizo una limpieza básica.
5. Guardo los documentos en formato JSON.

El corpus queda almacenado en Google Drive para que pueda ser utilizado por los
siguientes notebooks del proyecto.

Posteriormente utilizaré este corpus para realizar chunking, generar embeddings,
almacenarlos en Pinecone y construir el sistema RAG.

## Montaje de Google Drive

En esta celda monto mi Google Drive en Colab.

Utilizo una carpeta específica para el TP para que todos los notebooks puedan
compartir los mismos archivos de datos.

La estructura que voy a utilizar es:

TP_RAG_ARCA/
└── corpus/
    └── corpus_arca_monotributo.json

In [1]:
from google.colab import drive

drive.mount("/content/drive")

print("Google Drive montado correctamente.")

Mounted at /content/drive
Google Drive montado correctamente.


## Definición de las carpetas del proyecto

En esta celda defino una ubicación absoluta dentro de mi Google Drive.

De esta manera no dependo del directorio de trabajo actual de Colab.

Creo automáticamente la carpeta del proyecto y la carpeta donde voy a almacenar
el corpus.

In [2]:
from pathlib import Path

CARPETA_PROYECTO = Path(
    "/content/drive/MyDrive/TP_RAG_ARCA"
)

CARPETA_CORPUS = CARPETA_PROYECTO / "corpus"

CARPETA_PROYECTO.mkdir(
    parents=True,
    exist_ok=True
)

CARPETA_CORPUS.mkdir(
    parents=True,
    exist_ok=True
)

ARCHIVO_CORPUS = (
    CARPETA_CORPUS /
    "corpus_arca_monotributo.json"
)

print("Carpeta del proyecto:")
print(CARPETA_PROYECTO)

print("\nCarpeta del corpus:")
print(CARPETA_CORPUS)

print("\nArchivo del corpus:")
print(ARCHIVO_CORPUS)

Carpeta del proyecto:
/content/drive/MyDrive/TP_RAG_ARCA

Carpeta del corpus:
/content/drive/MyDrive/TP_RAG_ARCA/corpus

Archivo del corpus:
/content/drive/MyDrive/TP_RAG_ARCA/corpus/corpus_arca_monotributo.json


## Instalación de librerías

En esta celda instalo las librerías que necesito para descargar las páginas web
y extraer su contenido.

`requests` me permite realizar solicitudes HTTP.

`BeautifulSoup` me permite analizar el HTML y extraer el texto.

In [3]:
!pip install -q requests beautifulsoup4 lxml

## Importación de librerías

En esta celda importo las librerías que voy a utilizar durante el notebook.

También importo `datetime` para registrar la fecha y hora de descarga de cada
fuente.

In [4]:
import requests
import json

from bs4 import BeautifulSoup
from datetime import datetime, timezone

## Definición de las fuentes oficiales

En esta celda defino las páginas oficiales de ARCA que voy a utilizar como
corpus inicial.

Cada fuente tiene un identificador, un título y una URL.

Conservo estos datos porque posteriormente los utilizaré como metadatos de los
documentos y de los chunks del sistema RAG.

In [5]:
FUENTES_ARCA = [
    {
        "id": "inicio",
        "titulo": "Inicio - Ayuda sobre el Monotributo",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/inicio.asp"
    },
    {
        "id": "clave_fiscal",
        "titulo": "Obtención de Clave Fiscal",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/clave-fiscal.asp"
    },
    {
        "id": "constancias",
        "titulo": "Constancias y credenciales",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/constancias-y-credenciales.asp"
    },
    {
        "id": "facturacion",
        "titulo": "Facturación",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/facturacion.asp"
    },
    {
        "id": "recategorizacion",
        "titulo": "Recategorización",
        "url": "https://ftp.arca.gob.ar/monotributo/ayuda/recategorizacion.asp"
    },
    {
        "id": "baja",
        "titulo": "Baja de monotributo",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/baja.asp"
    },
    {
        "id": "desarrollo_actividad",
        "titulo": "Desarrollo de la actividad",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/desarrollo-de-la-actividad.asp"
    },
    {
        "id": "tutoriales",
        "titulo": "Tutoriales sobre Monotributo",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/tutoriales.asp"
    }
]

## Visualización de las fuentes

En esta celda verifico las fuentes que voy a descargar antes de comenzar el
proceso automático.

In [6]:
print("FUENTES DEL CORPUS")
print("=" * 80)

for i, fuente in enumerate(FUENTES_ARCA, start=1):
    print(f"\n{i}. {fuente['titulo']}")
    print(f"   ID: {fuente['id']}")
    print(f"   URL: {fuente['url']}")

FUENTES DEL CORPUS

1. Inicio - Ayuda sobre el Monotributo
   ID: inicio
   URL: https://www.arca.gob.ar/monotributo/ayuda/inicio.asp

2. Obtención de Clave Fiscal
   ID: clave_fiscal
   URL: https://www.arca.gob.ar/monotributo/ayuda/clave-fiscal.asp

3. Constancias y credenciales
   ID: constancias
   URL: https://www.arca.gob.ar/monotributo/ayuda/constancias-y-credenciales.asp

4. Facturación
   ID: facturacion
   URL: https://www.arca.gob.ar/monotributo/ayuda/facturacion.asp

5. Recategorización
   ID: recategorizacion
   URL: https://ftp.arca.gob.ar/monotributo/ayuda/recategorizacion.asp

6. Baja de monotributo
   ID: baja
   URL: https://www.arca.gob.ar/monotributo/ayuda/baja.asp

7. Desarrollo de la actividad
   ID: desarrollo_actividad
   URL: https://www.arca.gob.ar/monotributo/ayuda/desarrollo-de-la-actividad.asp

8. Tutoriales sobre Monotributo
   ID: tutoriales
   URL: https://www.arca.gob.ar/monotributo/ayuda/tutoriales.asp


## Función para descargar una página

En esta celda creo una función para descargar el contenido HTML de una URL.

Utilizo un `User-Agent` para identificar la solicitud como proveniente de un
cliente HTTP.

También utilizo `raise_for_status()` para detectar errores HTTP.

In [7]:
def descargar_pagina(url):

    headers = {
        "User-Agent": (
            "Mozilla/5.0 "
            "(compatible; RAG-ARCA-TP/1.0)"
        )
    }

    respuesta = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    respuesta.raise_for_status()

    return respuesta.text

## Prueba de descarga

En esta celda pruebo la descarga de una de las fuentes antes de procesar todo
el corpus.

In [8]:
html_prueba = descargar_pagina(
    FUENTES_ARCA[0]["url"]
)

print(
    f"HTML descargado correctamente: "
    f"{len(html_prueba)} caracteres"
)

print("\nPrimeros 1000 caracteres:\n")

print(html_prueba[:1000])

HTML descargado correctamente: 28464 caracteres

Primeros 1000 caracteres:

<!DOCTYPE html>
<html lang="es">

    <head>
        <meta charset="utf-8">
        <meta http-equiv="X-UA-Compatible" content="IE=edge">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <!-- Meta para los Buscadores -->
        <title>Inicio - Ayuda sobre el monotributo - Monotributo | ARCA</title>
        <meta name="description" content="Toda la informaciÃ³n sobre cÃ³mo darte de alta y realizar gestiones como monotributista">
        <meta name="keywords" content="monotributo, impositivo, montos, recategorizaciÃ³n, rÃ©gimen,">
        <meta name="author" content="ARCA">
        <meta name="robots" content="Index, Follow">
        <!-- opciones: Index, Follow - NoIndex, Follow - Index, NoFollow - NoIndex, NoFollow -->
        <!-- Facebook , google + -->
        <meta property="og:title" content="Inicio - Ayuda sobre el monotributo - Monotributo | ARCA">
        <meta prope

## Extracción del texto

En esta celda creo una función para convertir el HTML en texto.

Elimino elementos que no necesito para el corpus, como scripts, estilos,
iframes, SVG y contenido similar.

Después extraigo el texto visible y realizo una limpieza básica de espacios.

In [9]:
def extraer_texto(html):

    soup = BeautifulSoup(
        html,
        "lxml"
    )

    for elemento in soup([
        "script",
        "style",
        "noscript",
        "iframe",
        "svg"
    ]):
        elemento.decompose()

    texto = soup.get_text(
        separator="\n"
    )

    lineas = []

    for linea in texto.splitlines():

        linea_limpia = " ".join(
            linea.split()
        )

        if linea_limpia:
            lineas.append(
                linea_limpia
            )

    return "\n".join(lineas)

## Prueba de extracción

En esta celda verifico el texto obtenido a partir de la página descargada.

Esta inspección me permite comprobar que la extracción funciona antes de
procesar todas las fuentes.

In [10]:
texto_prueba = extraer_texto(
    html_prueba
)

print(
    f"Texto extraído: "
    f"{len(texto_prueba)} caracteres"
)

print("\nPrimeros 5000 caracteres:\n")

print(texto_prueba[:5000])

Texto extraído: 1994 caracteres

Primeros 5000 caracteres:

Inicio - Ayuda sobre el monotributo - Monotributo | ARCA
Evitar las herramientas de navegaciÃ³n y pasar al contenido
Monotributo
Menu
Inicio
Ayuda
Inicio
Ayuda sobre el monotributo
Inicio
Ayuda sobre el monotributo
Toda la informaciÃ³n sobre cÃ³mo darte de alta y hacer operaciones como monotributista.
Ingresar con clave fiscal
MenÃº de contenidos
QuÃ© es
INSCRIPCIÃN
Inicio
Clave fiscal
CUIT
Domicilio Fiscal ElectrÃ³nico
Jurisdicciones
Actividades
ALTA DE MONOTRIBUTO
Procedimiento
Tipos de monotributo
ParÃ¡metros
JubilaciÃ³n
Obra social
Monotributo unificado
Constancias y credenciales
DESPUÃS DEL ALTA
Desarrollo de la actividad
FacturaciÃ³n
Pagos
RecategorizaciÃ³n
FINALIZACIÃN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
ExclusiÃ³n
Renuncia
Pasaje al rÃ©gimen general
Ayuda
Inicio
El primer paso es inscribirse ante ARCA para poder despuÃ©s darse de alta en impuestos y utilizar los servicios con clave fiscal.
Para ell

## Descarga y procesamiento de todas las fuentes

En esta celda recorro todas las fuentes oficiales definidas anteriormente.

Para cada fuente:

1. Descargo la página.
2. Extraigo el texto.
3. Creo un documento estructurado.
4. Conservo los metadatos de la fuente.
5. Registro la fecha de descarga.

Si una fuente produce un error, lo informo y continúo con las restantes.

In [11]:
corpus = []

for fuente in FUENTES_ARCA:

    print("=" * 80)
    print(f"Procesando: {fuente['titulo']}")
    print(f"URL: {fuente['url']}")

    try:

        html = descargar_pagina(
            fuente["url"]
        )

        texto = extraer_texto(
            html
        )

        documento = {
            "id": fuente["id"],
            "titulo": fuente["titulo"],
            "organismo": "ARCA",
            "url": fuente["url"],
            "fecha_descarga": datetime.now(
                timezone.utc
            ).isoformat(),
            "texto": texto
        }

        corpus.append(
            documento
        )

        print(
            f"OK - {len(texto)} caracteres extraídos"
        )

    except Exception as error:

        print(
            f"ERROR: {error}"
        )

print("\n" + "=" * 80)
print("PROCESO TERMINADO")
print("=" * 80)

print(
    f"Documentos descargados correctamente: "
    f"{len(corpus)}"
)

Procesando: Inicio - Ayuda sobre el Monotributo
URL: https://www.arca.gob.ar/monotributo/ayuda/inicio.asp
OK - 1994 caracteres extraídos
Procesando: Obtención de Clave Fiscal
URL: https://www.arca.gob.ar/monotributo/ayuda/clave-fiscal.asp
OK - 2666 caracteres extraídos
Procesando: Constancias y credenciales
URL: https://www.arca.gob.ar/monotributo/ayuda/constancias-y-credenciales.asp
OK - 2626 caracteres extraídos
Procesando: Facturación
URL: https://www.arca.gob.ar/monotributo/ayuda/facturacion.asp
OK - 4305 caracteres extraídos
Procesando: Recategorización
URL: https://ftp.arca.gob.ar/monotributo/ayuda/recategorizacion.asp
OK - 4704 caracteres extraídos
Procesando: Baja de monotributo
URL: https://www.arca.gob.ar/monotributo/ayuda/baja.asp
OK - 2301 caracteres extraídos
Procesando: Desarrollo de la actividad
URL: https://www.arca.gob.ar/monotributo/ayuda/desarrollo-de-la-actividad.asp
OK - 2203 caracteres extraídos
Procesando: Tutoriales sobre Monotributo
URL: https://www.arca.gob.ar

## Resumen del corpus

En esta celda reviso el tamaño de cada documento descargado.

Busco detectar fuentes que tengan una cantidad de texto anormalmente pequeña.

In [12]:
print("RESUMEN DEL CORPUS")
print("=" * 80)

for documento in corpus:

    print(f"\nID: {documento['id']}")
    print(f"Título: {documento['titulo']}")
    print(f"Caracteres: {len(documento['texto'])}")
    print(f"URL: {documento['url']}")

RESUMEN DEL CORPUS

ID: inicio
Título: Inicio - Ayuda sobre el Monotributo
Caracteres: 1994
URL: https://www.arca.gob.ar/monotributo/ayuda/inicio.asp

ID: clave_fiscal
Título: Obtención de Clave Fiscal
Caracteres: 2666
URL: https://www.arca.gob.ar/monotributo/ayuda/clave-fiscal.asp

ID: constancias
Título: Constancias y credenciales
Caracteres: 2626
URL: https://www.arca.gob.ar/monotributo/ayuda/constancias-y-credenciales.asp

ID: facturacion
Título: Facturación
Caracteres: 4305
URL: https://www.arca.gob.ar/monotributo/ayuda/facturacion.asp

ID: recategorizacion
Título: Recategorización
Caracteres: 4704
URL: https://ftp.arca.gob.ar/monotributo/ayuda/recategorizacion.asp

ID: baja
Título: Baja de monotributo
Caracteres: 2301
URL: https://www.arca.gob.ar/monotributo/ayuda/baja.asp

ID: desarrollo_actividad
Título: Desarrollo de la actividad
Caracteres: 2203
URL: https://www.arca.gob.ar/monotributo/ayuda/desarrollo-de-la-actividad.asp

ID: tutoriales
Título: Tutoriales sobre Monotributo
C

## Visualización de un documento

En esta celda inspecciono manualmente uno de los documentos obtenidos.

Esta revisión me permite comprobar la calidad del contenido antes de guardarlo
como parte del corpus definitivo.

In [13]:
if not corpus:
    raise ValueError(
        "No se descargó ningún documento."
    )

documento = corpus[0]

print("=" * 80)
print(documento["titulo"])
print("=" * 80)

print(documento["texto"][:10000])

Inicio - Ayuda sobre el Monotributo
Inicio - Ayuda sobre el monotributo - Monotributo | ARCA
Evitar las herramientas de navegaciÃ³n y pasar al contenido
Monotributo
Menu
Inicio
Ayuda
Inicio
Ayuda sobre el monotributo
Inicio
Ayuda sobre el monotributo
Toda la informaciÃ³n sobre cÃ³mo darte de alta y hacer operaciones como monotributista.
Ingresar con clave fiscal
MenÃº de contenidos
QuÃ© es
INSCRIPCIÃN
Inicio
Clave fiscal
CUIT
Domicilio Fiscal ElectrÃ³nico
Jurisdicciones
Actividades
ALTA DE MONOTRIBUTO
Procedimiento
Tipos de monotributo
ParÃ¡metros
JubilaciÃ³n
Obra social
Monotributo unificado
Constancias y credenciales
DESPUÃS DEL ALTA
Desarrollo de la actividad
FacturaciÃ³n
Pagos
RecategorizaciÃ³n
FINALIZACIÃN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
ExclusiÃ³n
Renuncia
Pasaje al rÃ©gimen general
Ayuda
Inicio
El primer paso es inscribirse ante ARCA para poder despuÃ©s darse de alta en impuestos y utilizar los servicios con clave fiscal.
Para ello es necesario obtener l

## Guardado del corpus en Google Drive

En esta celda guardo el corpus en formato JSON.

Utilizo una ruta absoluta dentro de Google Drive, por lo que el archivo no
depende del directorio de trabajo de Colab.

El archivo queda guardado como:

`TP_RAG_ARCA/corpus/corpus_arca_monotributo.json`

Utilizo `ensure_ascii=False` para conservar correctamente caracteres como
acentos y ñ.

In [14]:
with open(
    ARCHIVO_CORPUS,
    "w",
    encoding="utf-8"
) as archivo:

    json.dump(
        corpus,
        archivo,
        ensure_ascii=False,
        indent=4
    )

print("Corpus guardado correctamente.")
print()
print("Ruta:")
print(ARCHIVO_CORPUS)
print()
print(
    f"Documentos guardados: {len(corpus)}"
)

Corpus guardado correctamente.

Ruta:
/content/drive/MyDrive/TP_RAG_ARCA/corpus/corpus_arca_monotributo.json

Documentos guardados: 8


## Verificación del archivo guardado

En esta celda vuelvo a abrir el archivo que acabo de guardar.

De esta manera compruebo que el JSON existe, puede ser leído correctamente y
contiene la cantidad esperada de documentos.

In [15]:
if not ARCHIVO_CORPUS.exists():

    raise FileNotFoundError(
        f"No se encontró el archivo después de guardarlo:\n"
        f"{ARCHIVO_CORPUS}"
    )

with open(
    ARCHIVO_CORPUS,
    "r",
    encoding="utf-8"
) as archivo:

    corpus_verificado = json.load(
        archivo
    )

print("Archivo verificado correctamente.")
print()
print(f"Ruta: {ARCHIVO_CORPUS}")
print(
    f"Documentos: {len(corpus_verificado)}"
)

print("\nDocumentos:")

for documento in corpus_verificado:

    print(
        f"- {documento['titulo']}"
    )

Archivo verificado correctamente.

Ruta: /content/drive/MyDrive/TP_RAG_ARCA/corpus/corpus_arca_monotributo.json
Documentos: 8

Documentos:
- Inicio - Ayuda sobre el Monotributo
- Obtención de Clave Fiscal
- Constancias y credenciales
- Facturación
- Recategorización
- Baja de monotributo
- Desarrollo de la actividad
- Tutoriales sobre Monotributo


## Conclusión

En este notebook construí el corpus documental inicial para mi proyecto RAG.

El corpus contiene información obtenida de fuentes oficiales de ARCA relacionadas
con el Monotributo.

Cada documento conserva:

- Un identificador.
- El título de la fuente.
- El organismo de origen.
- La URL oficial.
- La fecha de descarga.
- El texto extraído.

El corpus queda almacenado en Google Drive y será utilizado como entrada para
las siguientes etapas del proyecto.

En el próximo notebook voy a trabajar sobre la calidad y normalización del
texto antes de realizar el chunking y generar los embeddings.